<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/master/sigext_new_samples_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SigExt — New Sample Inference (no retraining)
Runs the full SigExt pipeline on a new set of samples using an already-trained Longformer checkpoint — no retraining needed.

Pipeline:
1. `prepare_data.py` — sample new examples from the dataset (different seed)
2. `inference_longformer_extractor.py` — run keyphrase extraction with the trained checkpoint
3. `zs_summarization.py` — generate summaries with the LLM + keyphrases

Outputs go to a separate directory so they can be used as an independent verification set.

In [ ]:
#@title Requirements
!pip install transformers datasets accelerate evaluate rouge_score torch jsonlines rake_nltk pytorch_lightning -q
!pip install rapidfuzz -q
!pip install boto3 -q
!pip install mistralai --upgrade -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.9/935.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the package

In [ ]:
#@title Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# @markdown Check this box to clone the repo from GitHub. Uncheck to load from Google Drive.

clone_repo = True # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"


Cloning into 'SigExt'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 115 (delta 61), reused 17 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 164.11 KiB | 1.69 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [ ]:
#@title Configuration

# path to the SigExt repo on your Drive
path = "/content/drive/MyDrive/ColabNotebooks/SigExt"  #@param{type:"string"}

# trained Longformer checkpoint (output of the main training notebook)
path_to_checkpoint = "experiments2/cnn_extractor_model/"  #@param{type:"string"}

# where to save the raw new samples
path_new_dataset    = "experiments/cnn_dataset_new_samples/"  #@param{type:"string"}

# where to save the samples after keyphrase extraction
path_new_with_kp    = "experiments/cnn_dataset_new_samples_with_keyphrase/"  #@param{type:"string"}

# where to save the final summaries
path_new_predictions = "experiments/cnn_extsig_predictions_new_samples_k15/"  #@param{type:"string"}

# sampling config
dataset     = "cnn"   #@param{type:"string"}
new_seed    = 123     #@param{type:"integer"}   # use a different seed from the original run
n_samples   = 500     #@param{type:"integer"}
top_k_kp    = 15      #@param{type:"integer"}

print(f"Repo path:          {path}")
print(f"Checkpoint:         {path}/{path_to_checkpoint}")
print(f"Raw samples:        {path}/{path_new_dataset}")
print(f"With keyphrases:    {path}/{path_new_with_kp}")
print(f"Predictions:        {path}/{path_new_predictions}")
print(f"Seed:               {new_seed}")
print(f"Samples:            {n_samples}")
print(f"Top-K keyphrases:   {top_k_kp}")


Repo path:          /content/drive/MyDrive/ColabNotebooks/SigExt
Checkpoint:         /content/drive/MyDrive/ColabNotebooks/SigExt/experiments2/cnn_extractor_model/
New raw dataset:    /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_new_samples/
New + keyphrases:   /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_new_samples_with_keyphrase/
New predictions:    /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_new_samples_k15/
Seed:               123
Samples:            500
Top-K keyphrases:   15


In [ ]:
#@title Setup
import os, sys

os.chdir(path)
print(f"Working directory: {os.getcwd()}")

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# make sure the checkpoint is there before running anything
ckpt_full = f"{path}/{path_to_checkpoint}"
if os.path.exists(ckpt_full):
    files = os.listdir(ckpt_full)
    print(f"\n Checkpoint found: {ckpt_full}")
    print(f"Files: {files[:5]}")
else:
    print(f"\n Checkpoint not found: {ckpt_full}")
    print("Run the main SigExt training notebook first.")


Working directory: /content/drive/.shortcut-targets-by-id/1TpffEVc0xhfd78UxnDSQbY7er61_yzCk/SigExt

 Checkpoint found: /content/drive/MyDrive/ColabNotebooks/SigExt/experiments2/cnn_extractor_model/
   Files: ['lightning_logs']


In [ ]:
# dataset to load from HuggingFace
hf_dataset_name    = "cnn_dailymail"   #@param{type:"string"}
hf_dataset_version = "3.0.0"          #@param{type:"string"}
hf_split           = "test"           #@param{type:"string"}

# field names in the raw dataset
input_field  = "article"     #@param{type:"string"}
output_field = "highlights"  #@param{type:"string"}

# original tuning set — used only for the overlap check
path_orig_dataset = "experiments/cnn_dataset_with_keyphrase/"  #@param{type:"string"}

In [ ]:
#@title Step 1 — Sample new examples

import os, sys, importlib, jsonlines, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
from functools import partial

dataset_out = f"{path}/{path_new_dataset}"
os.makedirs(dataset_out, exist_ok=True)

if os.path.exists(f"{dataset_out}test.jsonl"):
    with jsonlines.open(f"{dataset_out}test.jsonl") as f:
        existing = list(f)
    print(f"Already exists: {len(existing)} examples — skipping.")
else:
    os.chdir(path)
    if f"{path}/src" not in sys.path:
        sys.path.insert(0, f"{path}/src")
    import prepare_data
    importlib.reload(prepare_data)

    print(f"Loading {hf_dataset_name} ({hf_split} split)...")
    raw_dataset = load_dataset(hf_dataset_name, hf_dataset_version, split=hf_split)
    tokenizer   = AutoTokenizer.from_pretrained("allenai/longformer-large-4096")

    input_processor  = partial(prepare_data.general_input_processor,  field=input_field)
    output_processor = partial(prepare_data.general_output_processor, field=output_field)

    np.random.seed(new_seed)
    print(f"Sampling {n_samples} examples (seed={new_seed})...")

    processed = prepare_data.process_split(
        data             = raw_dataset,
        num_sample       = n_samples,
        input_processor  = input_processor,
        output_processor = output_processor,
        tokenizer        = tokenizer,
        max_input_len    = 6000,
        max_output_len   = 510,
    )

    with jsonlines.open(f"{dataset_out}test.jsonl", "w") as f:
        f.write_all(processed)
    print(f"Saved {len(processed)} examples to {dataset_out}test.jsonl")

# check there's no overlap with the original tuning set
new_file  = f"{dataset_out}test.jsonl"
orig_file = f"{path}/{path_orig_dataset}test.jsonl"

if os.path.exists(new_file) and os.path.exists(orig_file):
    with jsonlines.open(new_file) as f:
        new_data = list(f)
    with jsonlines.open(orig_file) as f:
        orig_data = list(f)

    new_idx  = set(d["index"] for d in new_data  if "index" in d)
    orig_idx = set(d["index"] for d in orig_data if "index" in d)
    overlap  = new_idx & orig_idx

    print(f"\nOverlap check:")
    print(f"New:      {len(new_data)}")
    print(f"Original: {len(orig_data)}")
    print(f"Overlap:  {len(overlap)}")
    if overlap:
        print(f"Try a different new_seed to avoid overlap.")
    else:
        print(f"No overlap — sets are independent.")
elif not os.path.exists(orig_file):
    print(f"\n Overlap check skipped — original tuning set not found at {orig_file}")

In [ ]:
import jsonlines
import os

out = f"{path}/{path_new_dataset}test.jsonl"
if os.path.exists(out):
    with jsonlines.open(out) as f:
        saved = list(f)
    print(f"Saved: {len(saved)} examples")
    print(f"Keys:  {list(saved[0].keys())}")
else:
    print("File not found")

Saved: 500 examples
Sample keys: ['raw_input', 'raw_output', 'trunc_input', 'trunc_output', 'input_length_info', 'output_length_info', 'trunc_input_phrases', 'trunc_output_phrases']


In [ ]:
#@title Step 2 — Extract keyphrases with trained Longformer checkpoint

import os

os.chdir(path)

ckpt_dir = f"{path}/experiments2/cnn_extractor_model"

kp_out   = f"{path}/{path_new_with_kp}"
data_in  = f"{path}/{path_new_dataset}"
os.makedirs(kp_out, exist_ok=True)

print(f"Checkpoint: {ckpt_dir}")
print(f"Input:      {data_in}")
print(f"Output:     {kp_out}")

if os.path.exists(f"{kp_out}test.jsonl"):
    import jsonlines
    with jsonlines.open(f"{kp_out}test.jsonl") as f:
        existing = list(f)
    print(f"\n Already extracted ({len(existing)} examples) — skipping.")
else:
    !python3 src/inference_longformer_extractor.py \
        --dataset_dir    {data_in} \
        --checkpoint_dir {ckpt_dir} \
        --output_dir     {kp_out}

# quick sanity check on the output
kp_file = f"{kp_out}test.jsonl"
if os.path.exists(kp_file):
    import jsonlines
    with jsonlines.open(kp_file) as f:
        kp_data = list(f)
    print(f"\nOutput check:")
    print(f"Examples:           {len(kp_data)}")
    print(f"Has input_kw_model: {'input_kw_model' in kp_data[0]}")
else:
    print(f"Output not found: {kp_file}")


Checkpoint: /content/drive/MyDrive/ColabNotebooks/SigExt/experiments2/cnn_extractor_model
Input:      /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_new_samples/
Output:     /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_new_samples_with_keyphrase/
INFO:root:Using f/content/drive/MyDrive/ColabNotebooks/SigExt/experiments2/cnn_extractor_model/lightning_logs/version_0/checkpoints/epoch_09-step_010000-recall20_0.200.ckpt
INFO:httpx:HTTP Request: HEAD https://huggingface.co/allenai/longformer-base-4096/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/allenai/longformer-base-4096/301e6a42cb0d9976a6d6a26a079fef81c18aa895/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/allenai/longformer-base-4096/301e6a42cb0d9976a6d6a26a079fef81c18aa895/config.json "HTTP/1.1 200 OK"
config.json: 100% 694/694 [00:00<00:00

In [ ]:
#@title Step 3 — Generate summaries (Mistral API)
# Reads keyphrases from path_new_with_kp and writes summaries to path_new_predictions.

import sys
import os
from types import ModuleType
import time

def chat(prompt, client):
    chat_response = client.chat.complete(
        model="mistral-small-2603",
        messages=[{"role": "user", "content": prompt["prompt_input"]}]
    )
    return chat_response.choices[0].message.content

def predict_one_eg_mistral(prompt):
    from mistralai.client import Mistral
    client = Mistral(api_key="YdbLhqEepgXav0x8ruTJiWR20GsgWCrD")
    err = None
    time.sleep(1.1)
    for i in range(5):
        try:
            return chat(prompt, client)
        except Exception as e:
            if "429" in str(e):
                print("Rate limit hit! Sleeping for 10 seconds...")
                time.sleep(10)
                err = e
    return f"Error: {err}"

# zs_summarization uses multiprocessing.Pool — replace with a sequential version
class FakePool:
    def __init__(self, processes=None): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
    def imap(self, func, iterable):
        return map(func, iterable)

import multiprocessing
multiprocessing.Pool = FakePool

# inject bedrock_utils so zs_summarization.py can import it without failing
m = ModuleType("bedrock_utils")
m.predict_one_eg_mistral = predict_one_eg_mistral
m.predict_one_eg_claude_instant = lambda x: "Not configured"
sys.modules["bedrock_utils"] = m

src_path = f"{path}/src"
if src_path not in sys.path:
    sys.path.append(src_path)

import importlib
import zs_summarization

os.chdir(path)

sys.argv = [
    "zs_summarization.py",
    "--model_name",         "mistral",
    "--kw_strategy",        "sigext_topk",
    "--kw_model_top_k",     str(top_k_kp),
    "--dataset",            dataset,
    "--dataset_dir",        path_new_with_kp,
    "--output_dir",         path_new_predictions,
    "--inference_on_split", "test",
]

print(f"Input:  {path}/{path_new_with_kp}")
print(f"Output: {path}/{path_new_predictions}")
print()

importlib.reload(zs_summarization)
zs_summarization.main()


In [ ]:
#@title Setup Bedrock

import os
import sys
import json
import time
import traceback
import logging
from types import ModuleType

import boto3.session as boto3_session
import botocore.config

AWS_ACCESS_KEY_ID     = "test" #@param{type:"string"}
AWS_SECRET_ACCESS_KEY = "test" #@param{type:"string"}
AWS_REGION            = "us-east-1" #@param{type:"string"}
MODEL_ID              = "mistral.mistral-7b-instruct-v0:2" #@param["mistral.mistral-7b-instruct-v0:2","anthropic.claude-3-haiku-20240307-v1:0","anthropic.claude-3-sonnet-20240229-v1:0"]

os.environ["AWS_ACCESS_KEY_ID"]     = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"]    = AWS_REGION


def _build_body(prompt_input):
    # evaluated at call time so MODEL_ID changes are always picked up
    if MODEL_ID.startswith("mistral."):
        return {
            "prompt":      f"[INST] {prompt_input} [/INST]",
            "max_tokens":  512,
            "temperature": 1.0,
            "top_p":       0.8,
            "top_k":       15,
        }
    elif MODEL_ID.startswith("anthropic."):
        return {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens":        512,
            "messages": [{"role": "user", "content": prompt_input}],
        }
    else:
        raise ValueError(f"Unknown model family: {MODEL_ID}")


def _parse_response(response_body):
    if MODEL_ID.startswith("mistral."):
        return response_body["outputs"][0]["text"]
    elif MODEL_ID.startswith("anthropic."):
        return response_body["content"][0]["text"]


def predict_one_eg(x):
    current_session = boto3_session.Session()
    bedrock = current_session.client(
        service_name  = "bedrock-runtime",
        region_name   = AWS_REGION,
        endpoint_url  = f"https://bedrock-runtime.{AWS_REGION}.amazonaws.com",
        config=botocore.config.Config(
            read_timeout    = 120,
            connect_timeout = 120,
            retries         = {"max_attempts": 5},
        ),
    )

    for attempt in range(10):
        try:
            response      = bedrock.invoke_model(
                modelId     = MODEL_ID,
                contentType = "application/json",
                accept      = "*/*",
                body        = json.dumps(_build_body(x["prompt_input"])),
            )
            response_body = json.loads(response.get("body").read())
            logging.info(response_body)
            return _parse_response(response_body)
        except Exception:
            traceback.print_exc()
            time.sleep(5)
    return ""


# sequential pool — zs_summarization uses multiprocessing.Pool internally
class FakePool:
    def __init__(self, processes=None): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
    def imap(self, func, iterable):
        return map(func, iterable)

import multiprocessing
multiprocessing.Pool = FakePool

# inject bedrock_utils so zs_summarization.py finds it at import time
m = ModuleType("bedrock_utils")
m.predict_one_eg_mistral        = predict_one_eg
m.predict_one_eg_claude_instant = predict_one_eg
sys.modules["bedrock_utils"]    = m

print(f"Backend ready — {MODEL_ID} ({AWS_REGION})")
print("Run the connection test cell to verify credentials before inference.")

Testing Bedrock connection...
Bedrock connected
Model: mistral.mistral-7b-instruct-v0:2
Region: us-east-1
Response: 'Hello. (That was one word, I promise!)'


In [ ]:
#@title Bedrock Connection Test

print(f"Testing connection...")
print(f"  Model:  {MODEL_ID}")
print(f"  Region: {AWS_REGION}")
print()

try:
    result = predict_one_eg({"prompt_input": "Say hello in one word."})

    if result and result.strip():
        print("✓  Connection successful")
    else:
        print("✗  Empty response — check credentials and region.")

except Exception as e:
    err = str(e)
    print(f"✗  Connection failed: {err}")
    print()
    if "UnrecognizedClientException" in err or "InvalidClientTokenId" in err:
        print("Invalid credentials — check AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY.")
    elif "AccessDeniedException" in err:
        print("Access denied — check that your IAM user has bedrock:InvokeModel permission.")
    elif "ResourceNotFoundException" in err:
        print(f"Model not found — verify the model ID is exactly: {MODEL_ID}")
    elif "EndpointResolutionError" in err or "Could not connect" in err:
        print(f"Cannot reach endpoint — check that {AWS_REGION} is correct for your account.")
    else:
        print("Unexpected error — check your AWS account and IAM permissions.")

Sending test request to Bedrock...
  Model:  mistral.mistral-7b-instruct-v0:2
  Region: us-west-2

✓  Connection successful
   Response: 'Hello. (That was one word, I promise!)'

Bedrock is ready. You can now run the full inference pipeline.


In [ ]:
#@title Import zs_summarization
import sys
import os

src_path = f"{path}/src"
if src_path not in sys.path:
    sys.path.append(src_path)

os.chdir(path)
print(f"Working directory: {os.getcwd()}")

import zs_summarization
print("zs_summarization imported")


Working directory: /content/drive/.shortcut-targets-by-id/1TpffEVc0xhfd78UxnDSQbY7er61_yzCk/SigExt
✓  zs_summarization imported


In [ ]:
#@title Run inference

import sys
import importlib
import os

os.chdir(path)

sys.argv = [
    "zs_summarization.py",
    "--model_name",         "mistral",
    "--kw_strategy",        "sigext_topk",
    "--kw_model_top_k",     str(top_k_kp),
    "--dataset",            dataset,
    "--dataset_dir",        path_new_with_kp,
    "--output_dir",         path_new_predictions,
    "--inference_on_split", "test",
]

print(f"Input:  {path}/{path_new_with_kp}")
print(f"Output: {path}/{path_new_predictions}")
print(f"Model:  {MODEL_ID}")
print()

importlib.reload(zs_summarization)
zs_summarization.main()


Starting inference on 500 samples...
  Input:  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_new_samples_with_keyphrase/
  Output: /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_new_samples_k15/
  Model:  mistral.mistral-7b-instruct-v0:2



100%|██████████| 500/500 [12:41<00:00,  1.52s/it]


In [ ]:
#@title Verify output
import json, os

pred_path    = f"{path}/{path_new_predictions}test_predictions.json"
metrics_path = f"{path}/{path_new_predictions}test_metrics.json"
dataset_path = f"{path}/{path_new_predictions}test_dataset.jsonl"

print("Output files:")
for fpath in [pred_path, metrics_path, dataset_path]:
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) if exists else 0
    status = f"✓  ({size:,} bytes)" if exists else "✗  NOT FOUND"
    print(f"  {os.path.basename(fpath)}  {status}")

if os.path.exists(pred_path):
    with open(pred_path) as f:
        preds = json.load(f)
    print(f"\nPredictions: {len(preds)}")
    print(f"Sample:\n  {preds[0][:150]}")

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f"\nROUGE metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")

# paths to plug into the hallucination filter notebook
print("\n" + "="*65)
print("Paths for the hallucination filter notebook:")
print(f"  path_predictions    = '{path}/{path_new_predictions}test_predictions.json'")
print(f"  path_scored_dataset = '{path}/{path_new_with_kp}test.jsonl'")
print(f"  path_full_dataset   = '{path}/{path_new_predictions}test_dataset.jsonl'")


Output files:
  test_predictions.json  ✓  (226,939 bytes)
  test_metrics.json  ✓  (293 bytes)
  test_dataset.jsonl  ✓  (15,429,056 bytes)

Total predictions: 500
Sample:
   A blundering boyfriend from East Sussex, Jake Boys, surprised his girlfriend Emily-Victoria Canham with tickets to see One Direction and McBusted in 

ROUGE metrics (new samples):
  rouge1p: 33.834
  rouge1r: 47.665
  rouge1f: 37.8096
  rouge2p: 12.0847
  rouge2r: 17.0201
  rouge2f: 13.4664
  rougeLp: 21.0427
  rougeLr: 29.8014
  rougeLf: 23.5351
  rougeLsump: 29.716
  rougeLsumr: 41.8791
  rougeLsumf: 33.2105
  gen_len: 80.048

These predictions are ready for the hallucination filter.
Use the following paths in the hallucination filter notebook:
  path_predictions    = '/content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions_new_samples_k15/test_predictions.json'
  path_scored_dataset = '/content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_new_samples_with_keyphrase/test.json